In [1]:
import numpy as np

from utils import (create_transition_open, get_complete_DR, make_maze,
                   evaluate_planning, evaluate_planning_barrier_crossing,
                   plan_with_D)

## Helper functions

In [14]:
def plan_with_absorbed_eigenbasis(U_os, C, R, T_new, goal_state, start_state, max_steps=500):
    """
    Plan using Woodbury update with eigenvalues absorbed into the basis.
    
    With U_os = V @ diag(sqrt(lambda)), we have D_os ≈ U_os @ U_os.T.
    
    The Woodbury correction (Eq. 2 from the paper) becomes:
    
        G = U_os + U_os @ phi_C @ A @ psi_R
        D_new = G @ U_os.T
        
    For planning we only need exp_v = D_new_nt @ t, which decomposes as:
    
        exp_v = U_os_nt @ (w_base + phi_C @ A @ z_barrier)
        
    where:
        w_base = U_os_nt.T @ t              (d-dim, base planning)
        z_barrier = psi_R @ w_base          (O-dim, barrier effect on goal paths)
        correction = phi_C @ A @ z_barrier  (d-dim, correction in feature space)
    
    No separate Lambda matrix anywhere.
    """
    N = U_os.shape[0]
    num_affected = C.shape[1]
    nonterminal = [s for s in range(N) if s != goal_state]
    nt_idx = np.array(nonterminal)

    # Project barrier matrices into absorbed eigenbasis
    phi_C = U_os.T @ C       # d x O
    psi_R = R @ U_os          # O x d

    # POR: O x O
    Z = psi_R @ phi_C
    A = np.linalg.inv(np.eye(num_affected) - Z)

    # Planning
    t = T_new[np.ix_(nonterminal, [goal_state])].flatten()
    U_os_nt = U_os[nt_idx]   # (N-1) x d

    w_base = U_os_nt.T @ t                        # d-dim
    z_barrier = psi_R @ w_base                     # O-dim
    w_total = w_base + phi_C @ A @ z_barrier       # d-dim

    exp_v = U_os_nt @ w_total
    exp_v = np.maximum(exp_v, 1e-10)
    v = np.log(exp_v)

    values = np.full(N, -np.inf)
    for i, s in enumerate(nonterminal):
        values[s] = v[i]
    values[goal_state] = 10.0

    state = start_state
    path = [state]
    for step in range(max_steps):
        if state == goal_state:
            return step, path
        neighbors = np.where(T_new[state] > 0)[0]
        if len(neighbors) == 0:
            break
        state = neighbors[np.argmax(values[neighbors])]
        path.append(state)
    return max_steps, path

def woodbury(T_old, D, T_new):
    """Performs woodbury update in full space (ground truth)."""
    diffs = T_old != T_new
    diff_rows, _ = np.where(diffs)
    delta_locs = np.unique(diff_rows)

    N = T_old.shape[0]
    R = (T_new - T_old)[delta_locs, :]
    C = np.eye(N)[:, delta_locs]

    Z = R @ D @ C
    A = np.linalg.inv(np.eye(Z.shape[0]) - Z)
    woodbury_update = D @ C @ A @ R @ D

    D_new = D + woodbury_update
    return D_new, R, C, delta_locs

def build_psi_R(w, delta_Phi, psi_R_barrier, barrier_idx, neighbor_idx, num_affected, d):
    """Build full H matrix: fixed barrier rows + parameterized neighbor rows."""
    psi_R = np.zeros((num_affected, d))
    psi_R[barrier_idx, :] = psi_R_barrier
    psi_R[neighbor_idx, :] = np.diag(w) @ delta_Phi.T
    return psi_R

#### Global variables

In [3]:
ROWS, COLS = 20, 20
NUM_STATES = ROWS * COLS
BARRIER_TYPE = 'H'
d = 400

# Version 1

### Step 1 : Setup everything - build open-space DR & place barrier

In [4]:
# --- Build open-space environment ---
T_os = create_transition_open(shape=(ROWS, COLS))
D_os = get_complete_DR(T=T_os)

# --- Build absorbed eigenbasis ---
eigenvalues, eigvecs = np.linalg.eigh(D_os)
idx = np.argsort(eigenvalues)[::-1][:d]
V = eigvecs[:, idx]
lam = eigenvalues[idx]
U_os = V * np.sqrt(lam)[np.newaxis, :]  # (N, d)

# --- Add barrier ---
T_new, barrier_states = make_maze(T_os, BARRIER_TYPE, ROWS, COLS)
D_new, R, C, delta_locs = woodbury(T_os, D_os, T_new)
num_affected = len(delta_locs)

In [5]:
print(f"Grid: {ROWS}x{COLS} = {NUM_STATES} states")
print(f"Eigenbasis dimension: d = {d}")
print(f"Number of affected states: {num_affected}")
print(f"Reconstruction error: {np.linalg.norm(U_os @ U_os.T - D_os) / np.linalg.norm(D_os):.2e}")

Grid: 20x20 = 400 states
Eigenbasis dimension: d = 400
Number of affected states: 32
Reconstruction error: 7.70e-02


### Step 2 : Build feature-space projections and W parameterization


In [6]:
# Exact projections (model-based, for verification)
phi_C = U_os.T @ C              # (d, num_affected) — G in the notes
psi_R_exact = R @ U_os          # (num_affected, d) — H in the notes

# Split affected states into barrier (fixed) vs neighbor (learnable)
is_barrier = np.array([loc in barrier_states for loc in delta_locs])
barrier_idx = np.where(is_barrier)[0]
neighbor_idx = np.where(~is_barrier)[0]
n_barrier = len(barrier_idx)
n_neighbors = len(neighbor_idx)

# --- Barrier states: H rows are fully known, no W to learn ---
psi_R_barrier = psi_R_exact[barrier_idx, :] # (n_barrier, d)

# --- Neighbor states: decompose R[b,:] = w_b * r_hat[b,:] ---
R_neighbors = R[neighbor_idx, :]            # (n_neighbors, NUM_STATES)
w_true = np.zeros(n_neighbors)
delta_Phi = np.zeros((d, n_neighbors))

for i in range(n_neighbors):
    row = R_neighbors[i, :]
    total_lost = np.sum(np.abs(row[row < 0]))  # total redirected mass
    w_true[i] = total_lost
    r_hat_i = row / total_lost              # normalized direction
    delta_Phi[:, i] = r_hat_i @ U_os        # feature-space direction

# --- Verify: parameterized H matches exact H ---
psi_R_reconstructed = np.zeros_like(psi_R_exact)
psi_R_reconstructed[barrier_idx, :] = psi_R_barrier
psi_R_reconstructed[neighbor_idx, :] = np.diag(w_true) @ delta_Phi.T

print(f"Total affected states: {num_affected}")
print(f"Barrier states (H fixed):      {n_barrier}")
print(f"Neighbor states (W learnable):  {n_neighbors}")
print(f"True weights (unique values):   {np.unique(np.round(w_true, 4))}")
print(f"||H_reconstructed - H_exact||:  {np.linalg.norm(psi_R_reconstructed - psi_R_exact):.2e}")

Total affected states: 32
Barrier states (H fixed):      10
Neighbor states (W learnable):  22
True weights (unique values):   [0.25]
||H_reconstructed - H_exact||:  1.41e-16


### Step 3 : Loss function and gradient descent

In [7]:
def compute_loss_and_grad(theta, theta_bar, phi_C, delta_Phi, psi_R_barrier,
                          barrier_idx, neighbor_idx, num_affected, d):
    """
    Target-separated loss (Eq. 16) and analytic gradient w.r.t. theta.
    
    theta:      (n_neighbors,) unconstrained params for online weights
    theta_bar:  (n_neighbors,) frozen target params
    phi_C:      (d, num_affected) = G = U_os^T @ C
    delta_Phi:  (d, n_neighbors) feature-space directions for neighbor states
    psi_R_barrier: (n_barrier, d) fixed H rows for barrier states
    
    Returns: (loss, grad_theta)
    """
    w = np.tanh(theta)
    w_bar = np.tanh(theta_bar)
    
    # Build full H (online) and H_bar (target)
    psi_R = build_psi_R(w, delta_Phi, psi_R_barrier, barrier_idx, neighbor_idx, num_affected, d)
    psi_R_bar = build_psi_R(w_bar, delta_Phi, psi_R_barrier, barrier_idx, neighbor_idx, num_affected, d)
    
    # Target operator: M_bar = (I_d - phi_C @ H_bar)^{-1}
    M_bar = np.linalg.inv(np.eye(d) - phi_C @ psi_R_bar)
    
    # Residual: should be zero at the correct W
    residual = np.eye(d) + phi_C @ psi_R @ M_bar - M_bar
    
    # Loss
    loss = np.sum(residual ** 2)
    
    # --- Analytic gradient ---
    # dL/dw_b = 2 * g_b^T @ Residual @ m_b
    # where g_b = phi_C[:, neighbor_idx[b]], m_b = M_bar^T @ delta_Phi[:, b]
    
    phi_C_neighbors = phi_C[:, neighbor_idx]          # (d, n_neighbors)
    M_bar_delta = M_bar.T @ delta_Phi                 # (d, n_neighbors)
    
    # Each row b of temp1 is g_b^T @ Residual
    temp1 = phi_C_neighbors.T @ residual              # (n_neighbors, d)
    
    # grad_w[b] = 2 * temp1[b, :] @ M_bar_delta[:, b]
    grad_w = 2.0 * np.sum(temp1 * M_bar_delta.T, axis=1)   # (n_neighbors,)
    
    # Chain rule through tanh
    grad_theta = grad_w * (1.0 - np.tanh(theta) ** 2)
    
    return loss, grad_theta

#### Check gradient works

In [8]:
np.random.seed(42)
theta_test = np.random.randn(n_neighbors) * 0.1
theta_bar_test = np.random.randn(n_neighbors) * 0.1

loss, grad_analytic = compute_loss_and_grad(
    theta_test, theta_bar_test, phi_C, delta_Phi, psi_R_barrier,
    barrier_idx, neighbor_idx, num_affected, d
)

# Finite difference gradient
eps = 1e-5
grad_fd = np.zeros_like(theta_test)
for b in range(n_neighbors):
    theta_plus = theta_test.copy()
    theta_plus[b] += eps
    loss_plus, _ = compute_loss_and_grad(
        theta_plus, theta_bar_test, phi_C, delta_Phi, psi_R_barrier,
        barrier_idx, neighbor_idx, num_affected, d
    )
    grad_fd[b] = (loss_plus - loss) / eps

max_diff = np.max(np.abs(grad_analytic - grad_fd))
rel_diff = np.linalg.norm(grad_analytic - grad_fd) / (np.linalg.norm(grad_analytic) + 1e-12)

print(f"Loss at test point: {loss:.6f}")
print(f"Max absolute gradient difference:  {max_diff:.2e}")
print(f"Relative gradient difference:      {rel_diff:.2e}")
print(f"\nFirst 5 components:")
print(f"Analytic: {grad_analytic[:5]}")
print(f"Finite-d: {grad_fd[:5]}")

Loss at test point: 0.524927
Max absolute gradient difference:  2.14e-05
Relative gradient difference:      4.46e-05

First 5 components:
Analytic: [0.12640294 0.45571692 0.37524312 0.59873994 0.35203735]
Finite-d: [0.12642203 0.45573474 0.37526138 0.59875756 0.35205529]


### Step 4 : Training loop

In [9]:
def train_W(theta_init, phi_C, delta_Phi, psi_R_barrier,
            barrier_idx, neighbor_idx, num_affected, d, w_true,
            lr=0.01, n_iters=2000, target_update_freq=50):
    """
    Learn W via gradient descent on the target-separated loss (Eq. 16).
    
    Returns: dict with training history.
    """
    theta = theta_init.copy()
    theta_bar = theta.copy()
    
    history = {
        'loss': [],
        'w_error': [],         # ||tanh(theta) - w_true||
        'w_max_error': [],     # max_b |tanh(theta_b) - w_true_b|
        'w_learned': [],
    }
    
    for t in range(n_iters):
        loss, grad = compute_loss_and_grad(
            theta, theta_bar, phi_C, delta_Phi, psi_R_barrier,
            barrier_idx, neighbor_idx, num_affected, d
        )
        
        theta -= lr * grad
        
        # Sync target periodically
        if (t + 1) % target_update_freq == 0:
            theta_bar = theta.copy()
        
        w_learned = np.tanh(theta)
        history['loss'].append(loss)
        history['w_error'].append(np.linalg.norm(w_learned - w_true))
        history['w_max_error'].append(np.max(np.abs(w_learned - w_true)))
        history['w_learned'].append(w_learned.copy())
    
    return theta, history


# --- Run training --
theta_init = np.zeros(n_neighbors)
theta_learned, history = train_W(
    theta_init, phi_C, delta_Phi, psi_R_barrier,
    barrier_idx, neighbor_idx, num_affected, d, w_true,
    lr=0.01, n_iters=2000, target_update_freq=50
)

w_learned = np.tanh(theta_learned)

print(f"True W:    {w_true[:5]}  (all {w_true[0]:.4f})")
print(f"Learned W: {np.round(w_learned[:5], 4)}")
print(f"\nFinal loss:         {history['loss'][-1]:.2e}")
print(f"Final ||w - w_true||: {history['w_error'][-1]:.2e}")
print(f"Final max |w_b - w_true_b|: {history['w_max_error'][-1]:.2e}")

True W:    [0.25 0.25 0.25 0.25 0.25]  (all 0.2500)
Learned W: [-0. -0. -0. -0. -0.]

Final loss:         1.92e-29
Final ||w - w_true||: 1.17e+00
Final max |w_b - w_true_b|: 2.50e-01


In [10]:
# --- Run training with random initialization ---
np.random.seed(42)
theta_init = np.random.randn(n_neighbors) * 0.5

theta_learned, history = train_W(
    theta_init, phi_C, delta_Phi, psi_R_barrier,
    barrier_idx, neighbor_idx, num_affected, d, w_true,
    lr=0.01, n_iters=2000, target_update_freq=50
)

w_learned = np.tanh(theta_learned)

print(f"True W:    {w_true[:5]}  (all {w_true[0]:.4f})")
print(f"Learned W: {np.round(w_learned[:5], 4)}")
print(f"\nFinal loss:         {history['loss'][-1]:.2e}")
print(f"Final ||w - w_true||: {history['w_error'][-1]:.2e}")
print(f"Final max |w_b - w_true_b|: {history['w_max_error'][-1]:.2e}")
print(f"\nInitial W: {np.round(np.tanh(np.random.RandomState(42).randn(n_neighbors) * 0.5)[:5], 4)}")

True W:    [0.25 0.25 0.25 0.25 0.25]  (all 0.2500)
Learned W: [ 0.2434 -0.069   0.313   0.642  -0.1165]

Final loss:         2.81e-29
Final ||w - w_true||: 2.36e+00
Final max |w_b - w_true_b|: 9.93e-01

Initial W: [ 0.2434 -0.069   0.313   0.642  -0.1165]


# Version 2 : Prediction error approach

### Step 1: Precompute expected next-state features under T_old

In [11]:
# For each changed neighbor state, precompute what the OLD model
# predicted as the expected next-state feature vector.
# This is: T_old[s_b, :] @ U_os  (a d-dimensional vector)

# We only estimate H for the neighbor states (not barrier states,
# whose H rows are already known exactly)
neighbor_locs = delta_locs[neighbor_idx]  # actual state indices
barrier_locs = delta_locs[barrier_idx]

# Expected next-state features under old model for each neighbor state
expected_features_old = np.zeros((n_neighbors, d))
for i, s in enumerate(neighbor_locs):
    expected_features_old[i, :] = T_os[s, :] @ U_os

# Verify: the true H for neighbor states should equal
# E_{T_new}[u(s')] - E_{T_old}[u(s')]  =  (T_new - T_old)[s, :] @ U_os
expected_features_new = np.zeros((n_neighbors, d))
for i, s in enumerate(neighbor_locs):
    expected_features_new[i, :] = T_new[s, :] @ U_os

H_true_neighbors = expected_features_new - expected_features_old
print(f"||H_true_neighbors - psi_R_exact[neighbor_idx]||: "
      f"{np.linalg.norm(H_true_neighbors - psi_R_exact[neighbor_idx]):.2e}")

||H_true_neighbors - psi_R_exact[neighbor_idx]||: 3.77e-16


### Step 2: Estimate H from sampled transitions

In [12]:
def estimate_H_from_transitions(T_new, U_os, neighbor_locs, expected_features_old,
                                 psi_R_barrier, barrier_idx, neighbor_idx,
                                 num_affected, d, n_visits_per_state):
    """
    Simulate an agent visiting each changed neighbor state n_visits_per_state times,
    sampling a transition from T_new each time, and averaging prediction errors
    to estimate H.
    
    Returns: H_hat (num_affected, d), history of H_error after each visit
    """
    n_neighbors = len(neighbor_locs)
    H_hat_neighbors = np.zeros((n_neighbors, d))
    visit_counts = np.zeros(n_neighbors, dtype=int)
    
    history = {
        'H_error': [],      # ||H_hat - H_true|| after each visit
        'visit': [],         # cumulative visit number
    }
    
    rng = np.random.RandomState(42)
    visit_num = 0
    
    for _ in range(n_visits_per_state):
        # Visit each changed state once (random order)
        order = rng.permutation(n_neighbors)
        for i in order:
            s = neighbor_locs[i]
            
            # Sample a transition from T_new
            s_next = rng.choice(NUM_STATES, p=T_new[s, :])
            
            # Prediction error: where I went vs where old model expected
            err = U_os[s_next, :] - expected_features_old[i, :]
            
            # Running average update
            visit_counts[i] += 1
            H_hat_neighbors[i, :] += (err - H_hat_neighbors[i, :]) / visit_counts[i]
            
            # Log
            visit_num += 1
            H_full = np.zeros((num_affected, d))
            H_full[barrier_idx, :] = psi_R_barrier
            H_full[neighbor_idx, :] = H_hat_neighbors
            
            h_err = np.linalg.norm(H_full - psi_R_exact)
            history['H_error'].append(h_err)
            history['visit'].append(visit_num)
    
    # Assemble full H
    H_hat = np.zeros((num_affected, d))
    H_hat[barrier_idx, :] = psi_R_barrier
    H_hat[neighbor_idx, :] = H_hat_neighbors
    
    return H_hat, history


# === Run estimation at different sample counts ===
for n_visits in [1, 5, 10, 50, 100]:
    H_hat, hist = estimate_H_from_transitions(
        T_new, U_os, neighbor_locs, expected_features_old,
        psi_R_barrier, barrier_idx, neighbor_idx,
        num_affected, d, n_visits_per_state=n_visits
    )
    
    h_err = np.linalg.norm(H_hat - psi_R_exact) / np.linalg.norm(psi_R_exact)
    print(f"  {n_visits:>3d} visits/state ({n_visits * n_neighbors:>5d} total samples): "
          f"relative H error = {h_err:.4f}")

    1 visits/state (   22 total samples): relative H error = 1.4902
    5 visits/state (  110 total samples): relative H error = 0.6455
   10 visits/state (  220 total samples): relative H error = 0.4137
   50 visits/state ( 1100 total samples): relative H error = 0.2165
  100 visits/state ( 2200 total samples): relative H error = 0.1322


### Step 3a: From estimated H to M to planning

In [17]:
def compute_M_from_H(H_hat, phi_C, d):
    """Compute M = I + phi_C @ A @ H via exact inverse (Eq. 9)."""
    B = H_hat.shape[0]
    Z = H_hat @ phi_C          # B x B
    A = np.linalg.inv(np.eye(B) - Z)
    M = np.eye(d) + phi_C @ A @ H_hat
    return M


# Ground truth M from exact H
M_true = compute_M_from_H(psi_R_exact, phi_C, d)

# M with no repair
M_identity = np.eye(d)

print("Planning: state 0 -> state 399")
print("=" * 65)

# Ground truth baselines
steps_true_D, _ = plan_with_D(D_new, T_new, NUM_STATES - 1, 0)
steps_no_repair, _ = plan_with_D(D_os, T_new, NUM_STATES - 1, 0)
print(f"  Full Woodbury (ground truth D_new):   {steps_true_D} steps")
print(f"  No repair (old D_os):                 {steps_no_repair} steps")

# Plan with exact M
steps_exact_M, _ = plan_with_absorbed_eigenbasis(U_os, C, R, T_new, NUM_STATES - 1, 0)
print(f"  Exact M (model-based):                {steps_exact_M} steps")
print()

# Plan with estimated H at different sample counts
print("Estimated H -> M -> Planning:")
for n_visits in [1, 5, 10, 50, 100, 500]:
    H_hat, hist = estimate_H_from_transitions(
        T_new, U_os, neighbor_locs, expected_features_old,
        psi_R_barrier, barrier_idx, neighbor_idx,
        num_affected, d, n_visits_per_state=n_visits
    )
    
    M_hat = compute_M_from_H(H_hat, phi_C, d)
    
    # Plan using M_hat
    steps_hat, _ = plan_with_absorbed_eigenbasis(
        U_os, C, R, T_new, NUM_STATES - 1, 0
    )
    
    # We need to plan with our M_hat, not the model-based one.
    # Use the same planning logic but with our estimated M:
    goal = NUM_STATES - 1
    start = 0
    N = U_os.shape[0]
    nonterminal = [s for s in range(N) if s != goal]
    nt_idx = np.array(nonterminal)
    
    t_vec = T_new[np.ix_(nt_idx, [goal])].flatten()
    U_nt = U_os[nt_idx]
    
    w_base = U_nt.T @ t_vec
    w_total = M_hat @ w_base
    exp_v = U_nt @ w_total
    exp_v = np.maximum(exp_v, 1e-10)
    v = np.log(exp_v)
    
    values = np.full(N, -np.inf)
    for i, s in enumerate(nonterminal):
        values[s] = v[i]
    values[goal] = 10.0
    
    state = start
    path = [state]
    for _ in range(500):
        if state == goal:
            break
        neighbors = np.where(T_new[state] > 0)[0]
        if len(neighbors) == 0:
            break
        state = neighbors[np.argmax(values[neighbors])]
        path.append(state)
    steps_hat = len(path) - 1 if state == goal else 500
    
    h_err = np.linalg.norm(H_hat - psi_R_exact) / np.linalg.norm(psi_R_exact)
    m_err = np.linalg.norm(M_hat - M_true) / np.linalg.norm(M_true)
    
    print(f"  {n_visits:>3d} visits/state: H_err={h_err:.4f}, M_err={m_err:.4f}, "
          f"steps={steps_hat}")

Planning: state 0 -> state 399
  Full Woodbury (ground truth D_new):   38 steps
  No repair (old D_os):                 38 steps
  Exact M (model-based):                38 steps

Estimated H -> M -> Planning:
    1 visits/state: H_err=1.4902, M_err=0.5398, steps=38
    5 visits/state: H_err=0.6455, M_err=0.1183, steps=38
   10 visits/state: H_err=0.4137, M_err=0.0783, steps=38
   50 visits/state: H_err=0.2165, M_err=0.0420, steps=38
  100 visits/state: H_err=0.1322, M_err=0.0256, steps=38
  500 visits/state: H_err=0.0619, M_err=0.0110, steps=38


### Step 3b: Evaluate on barrier-crossing paths

In [18]:
def plan_with_M(U_os, M, T_new, goal, start, max_steps=500):
    """Plan using a given M operator in feature space."""
    N = U_os.shape[0]
    nonterminal = [s for s in range(N) if s != goal]
    nt_idx = np.array(nonterminal)
    
    t_vec = T_new[np.ix_(nt_idx, [goal])].flatten()
    U_nt = U_os[nt_idx]
    
    w_base = U_nt.T @ t_vec
    w_total = M @ w_base
    exp_v = U_nt @ w_total
    exp_v = np.maximum(exp_v, 1e-10)
    v = np.log(exp_v)
    
    values = np.full(N, -np.inf)
    for i, s in enumerate(nonterminal):
        values[s] = v[i]
    values[goal] = 10.0
    
    state = start
    path = [state]
    for _ in range(max_steps):
        if state == goal:
            break
        neighbors = np.where(T_new[state] > 0)[0]
        if len(neighbors) == 0:
            break
        state = neighbors[np.argmax(values[neighbors])]
        path.append(state)
    
    steps = len(path) - 1 if state == goal else max_steps
    return steps, path


def evaluate_barrier_crossing(U_os, M, T_new, barrier_states, 
                               rows, cols, n_trials=200, seed=42):
    """
    Evaluate planning on start-goal pairs that must cross the barrier.
    Returns mean steps, median steps, success rate.
    """
    N = rows * cols
    barrier_row = rows // 2
    valid_states = [s for s in range(N) if s not in barrier_states]
    rng = np.random.RandomState(seed)
    
    steps_list = []
    n_tested = 0
    
    while n_tested < n_trials:
        s = rng.choice(valid_states)
        g = rng.choice(valid_states)
        if s == g:
            continue
        s_row, g_row = s // cols, g // cols
        # Must cross barrier
        if not ((s_row < barrier_row and g_row >= barrier_row) or
                (s_row >= barrier_row and g_row < barrier_row)):
            continue
        
        st, _ = plan_with_M(U_os, M, T_new, g, s)
        steps_list.append(st)
        n_tested += 1
    
    steps_arr = np.array(steps_list)
    success = np.sum(steps_arr < 500) / n_trials
    return np.mean(steps_arr), np.median(steps_arr), success


# === Ground truth baselines ===
print("Barrier-crossing paths (200 random trials)")
print("=" * 65)

# Ground truth: plan with D_new
mean_gt, med_gt, suc_gt = evaluate_barrier_crossing(
    U_os, M_true, T_new, barrier_states, ROWS, COLS)
print(f"  Exact M (model-based):  mean={mean_gt:.1f}, median={med_gt:.1f}, success={suc_gt:.2f}")

# No repair
mean_nr, med_nr, suc_nr = evaluate_barrier_crossing(
    U_os, M_identity, T_new, barrier_states, ROWS, COLS)
print(f"  No repair (M=I):        mean={mean_nr:.1f}, median={med_nr:.1f}, success={suc_nr:.2f}")
print()

# === Estimated H at different sample counts ===
print("Estimated H -> M -> Planning (barrier-crossing):")
for n_visits in [1, 5, 10, 50, 100, 500]:
    H_hat, _ = estimate_H_from_transitions(
        T_new, U_os, neighbor_locs, expected_features_old,
        psi_R_barrier, barrier_idx, neighbor_idx,
        num_affected, d, n_visits_per_state=n_visits
    )
    M_hat = compute_M_from_H(H_hat, phi_C, d)
    
    mean_s, med_s, suc_s = evaluate_barrier_crossing(
        U_os, M_hat, T_new, barrier_states, ROWS, COLS)
    h_err = np.linalg.norm(H_hat - psi_R_exact) / np.linalg.norm(psi_R_exact)
    
    print(f"  {n_visits:>3d} visits/state ({n_visits * n_neighbors:>5d} samples): "
          f"mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}, H_err={h_err:.4f}")

Barrier-crossing paths (200 random trials)
  Exact M (model-based):  mean=18.3, median=18.0, success=1.00
  No repair (M=I):        mean=184.5, median=23.0, success=0.66

Estimated H -> M -> Planning (barrier-crossing):
    1 visits/state (   22 samples): mean=18.5, median=18.0, success=1.00, H_err=1.4902
    5 visits/state (  110 samples): mean=18.4, median=18.0, success=1.00, H_err=0.6455
   10 visits/state (  220 samples): mean=18.3, median=18.0, success=1.00, H_err=0.4137
   50 visits/state ( 1100 samples): mean=18.3, median=18.0, success=1.00, H_err=0.2165
  100 visits/state ( 2200 samples): mean=18.3, median=18.0, success=1.00, H_err=0.1322
  500 visits/state (11000 samples): mean=18.3, median=18.0, success=1.00, H_err=0.0619


# Version 3 : Self-consistency loss approach

In [19]:
# === Test advisor's alternating approach ===

def compute_loss_and_grad_fixed_M(theta, M_fixed, phi_C, delta_Phi, psi_R_barrier,
                                   barrier_idx, neighbor_idx, num_affected, d):
    """
    Loss with M held completely fixed (not derived from theta at all).
    L = ||I + G @ H(W(theta)) @ M_fixed - M_fixed||^2
    """
    w = np.tanh(theta)
    psi_R = build_psi_R(w, delta_Phi, psi_R_barrier, barrier_idx, neighbor_idx, num_affected, d)
    
    residual = np.eye(d) + phi_C @ psi_R @ M_fixed - M_fixed
    loss = np.sum(residual ** 2)
    
    # Gradient: same formula but M_bar replaced by M_fixed
    phi_C_neighbors = phi_C[:, neighbor_idx]
    M_fixed_delta = M_fixed.T @ delta_Phi
    temp1 = phi_C_neighbors.T @ residual
    grad_w = 2.0 * np.sum(temp1 * M_fixed_delta.T, axis=1)
    grad_theta = grad_w * (1.0 - np.tanh(theta) ** 2)
    
    return loss, grad_theta


def train_alternating(phi_C, delta_Phi, psi_R_barrier,
                      barrier_idx, neighbor_idx, num_affected, d, w_true,
                      n_outer=20, n_inner=100, lr=0.01):
    """
    Advisor's alternating scheme:
    1. Fix M, optimize W via gradient descent on ||I + GH(W)M - M||^2
    2. Build H from optimized W, set M = I + GH (one fixed-point step)
    3. Repeat
    """
    n_neighbors = len(w_true)
    theta = np.zeros(n_neighbors)
    M = np.eye(d)  # start with identity (no correction)
    
    history = {
        'w_error': [],
        'loss': [],
        'outer_iter': [],
    }
    
    for outer in range(n_outer):
        # Step 1: Fix M, optimize W
        for inner in range(n_inner):
            loss, grad = compute_loss_and_grad_fixed_M(
                theta, M, phi_C, delta_Phi, psi_R_barrier,
                barrier_idx, neighbor_idx, num_affected, d
            )
            theta -= lr * grad
        
        w_learned = np.tanh(theta)
        w_err = np.linalg.norm(w_learned - w_true)
        
        # Step 2: Build H from current W, set M = I + GH (one Neumann step)
        psi_R = build_psi_R(w_learned, delta_Phi, psi_R_barrier,
                            barrier_idx, neighbor_idx, num_affected, d)
        GH = phi_C @ psi_R
        M = np.eye(d) + GH  # NOT the exact inverse
        
        history['w_error'].append(w_err)
        history['loss'].append(loss)
        history['outer_iter'].append(outer)
        
        print(f"  Outer {outer+1:>2d}: loss={loss:.2e}, ||w - w_true||={w_err:.4f}, "
              f"w_mean={np.mean(w_learned):.4f}, w_std={np.std(w_learned):.4f}")
    
    return theta, M, history


# === Run it ===
print("Advisor's alternating approach:")
print("=" * 65)
theta_alt, M_alt, hist_alt = train_alternating(
    phi_C, delta_Phi, psi_R_barrier,
    barrier_idx, neighbor_idx, num_affected, d, w_true,
    n_outer=20, n_inner=100, lr=0.01
)

w_alt = np.tanh(theta_alt)
print(f"\nTrue W:    {w_true[:5]}")
print(f"Learned W: {np.round(w_alt[:5], 4)}")

# === Evaluate planning ===
# Build final M from learned W using exact inverse (for fair comparison)
psi_R_final = build_psi_R(w_alt, delta_Phi, psi_R_barrier,
                           barrier_idx, neighbor_idx, num_affected, d)
M_final = compute_M_from_H(psi_R_final, phi_C, d)

mean_s, med_s, suc_s = evaluate_barrier_crossing(
    U_os, M_final, T_new, barrier_states, ROWS, COLS)
print(f"\nBarrier-crossing planning: mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")

Advisor's alternating approach:
  Outer  1: loss=9.88e+00, ||w - w_true||=1.5803, w_mean=-0.0866, w_std=0.0147
  Outer  2: loss=1.74e+01, ||w - w_true||=2.4364, w_mean=-0.2651, w_std=0.0670
  Outer  3: loss=1.47e+01, ||w - w_true||=2.9460, w_mean=-0.3714, w_std=0.0915
  Outer  4: loss=1.88e+01, ||w - w_true||=3.2297, w_mean=-0.4318, w_std=0.0966
  Outer  5: loss=2.28e+01, ||w - w_true||=3.3842, w_mean=-0.4657, w_std=0.0915
  Outer  6: loss=2.54e+01, ||w - w_true||=3.4707, w_mean=-0.4854, w_std=0.0822
  Outer  7: loss=2.68e+01, ||w - w_true||=3.5224, w_mean=-0.4975, w_std=0.0717
  Outer  8: loss=2.77e+01, ||w - w_true||=3.5558, w_mean=-0.5056, w_std=0.0614
  Outer  9: loss=2.82e+01, ||w - w_true||=3.5792, w_mean=-0.5113, w_std=0.0522
  Outer 10: loss=2.85e+01, ||w - w_true||=3.5969, w_mean=-0.5156, w_std=0.0445
  Outer 11: loss=2.87e+01, ||w - w_true||=3.6108, w_mean=-0.5188, w_std=0.0390
  Outer 12: loss=2.88e+01, ||w - w_true||=3.6223, w_mean=-0.5214, w_std=0.0363
  Outer 13: loss=2.9

# Version 4 : Prediction error approach without T

In [20]:
# === Verify T-free prediction error approach ===

# Compute mu: eigenvalues of T from eigenvalues of D
# D = (e^c I - T)^{-1} with e^c = 1/gamma
# => T has same eigenvectors, with eigenvalues mu_i = 1/gamma - 1/lambda_i
gamma = 1 / (np.diag(np.linalg.inv(D_os) + T_os)[0])  # recover gamma if needed
# Or just use the known relationship:
# D_os = ((1/gamma)*I - T)^{-1}, eigenvalues of D are 1/(1/gamma - mu_i)
# => mu_i = 1/gamma - 1/lambda_i

mu = (1.0 / 0.9) - 1.0 / lam  # adjust 0.9 to your actual gamma

# Actually let's just recover gamma from the DR definition
# D = (e^c I - T)^{-1} means D^{-1} = e^c I - T, so T = e^c I - D^{-1}
# For eigenvalues: mu_i = e^c - 1/lambda_i
# We need to know e^c. From your get_complete_DR, what gamma do you use?

# Let's figure it out empirically: T has row sums = 1,
# so e^c = (1 + sum of 1/lambda_i * v_i ... ) 
# Easier: just check what gamma gives us the right T

# Direct verification: compute T_old @ U_os both ways
# Method 1: using T_old explicitly
predicted_with_T = np.zeros((n_neighbors, d))
for i, s in enumerate(neighbor_locs):
    predicted_with_T[i, :] = T_os[s, :] @ U_os

# Method 2: using eigenvalues only
# First we need the correct e^c. Since D = (e^c I - T)^{-1},
# and D has eigenvalues lam_i, T has eigenvalues e^c - 1/lam_i.
# T is row-stochastic so its largest eigenvalue is 1.
# The largest eigenvalue of D corresponds to the largest lam.
# So: 1 = e^c - 1/lam_max => e^c = 1 + 1/lam_max

ec = 1.0 + 1.0 / lam[0]  # lam is sorted descending, so lam[0] is largest
mu = ec - 1.0 / lam       # eigenvalues of T, shape (d,)

print(f"Recovered e^c = {ec:.6f}")
print(f"mu range: [{mu.min():.4f}, {mu.max():.4f}] (should include 1.0)")

# Method 2: T_old[s,:] @ U_os = u(s) * mu  (element-wise)
predicted_without_T = np.zeros((n_neighbors, d))
for i, s in enumerate(neighbor_locs):
    predicted_without_T[i, :] = U_os[s, :] * mu

# Compare
diff = np.linalg.norm(predicted_with_T - predicted_without_T)
print(f"\n||T_old @ U_os (explicit) - u(s)*mu (T-free)||: {diff:.2e}")

Recovered e^c = 1.104225
mu range: [-1.8036, 1.0000] (should include 1.0)

||T_old @ U_os (explicit) - u(s)*mu (T-free)||: 1.22e-02


In [21]:
# === Verify: is the error just the eigenbasis approximation error? ===

# Check if any eigenvalues are negative or near-zero
print(f"Eigenvalue range: [{lam.min():.6f}, {lam.max():.6f}]")
print(f"Number of negative eigenvalues: {np.sum(lam < 0)}")
print(f"Number of eigenvalues < 1e-6:   {np.sum(lam < 1e-6)}")
print(f"D_os reconstruction error:       {np.linalg.norm(U_os @ U_os.T - D_os) / np.linalg.norm(D_os):.2e}")

# The T-free error should be on the same order as the reconstruction error.
# Let's check: compare both prediction methods against the TRUE H
# to see if the T-free version is any worse for the thing we actually care about.

# H estimated via T_old explicitly
H_with_T = np.zeros((n_neighbors, d))
for i, s in enumerate(neighbor_locs):
    H_with_T[i, :] = T_new[s, :] @ U_os - T_os[s, :] @ U_os

# H estimated via T-free predicted features
H_without_T = np.zeros((n_neighbors, d))
for i, s in enumerate(neighbor_locs):
    predicted_Tfree = U_os[s, :] * mu
    H_without_T[i, :] = T_new[s, :] @ U_os - predicted_Tfree

print(f"\n||H_with_T - H_exact||:    {np.linalg.norm(H_with_T - psi_R_exact[neighbor_idx]):.2e}")
print(f"||H_without_T - H_exact||: {np.linalg.norm(H_without_T - psi_R_exact[neighbor_idx]):.2e}")

# Now the real test: does the T-free version still plan well?
# Run the same estimation loop but using u(s)*mu instead of T_old[s,:]@U_os

def estimate_H_Tfree(T_new, U_os, mu, neighbor_locs, 
                     psi_R_barrier, barrier_idx, neighbor_idx,
                     num_affected, d, n_visits_per_state):
    """Same as before but predicted features use u(s)*mu instead of T_old@U_os."""
    n_neighbors = len(neighbor_locs)
    H_hat_neighbors = np.zeros((n_neighbors, d))
    visit_counts = np.zeros(n_neighbors, dtype=int)
    
    rng = np.random.RandomState(42)
    
    for _ in range(n_visits_per_state):
        order = rng.permutation(n_neighbors)
        for i in order:
            s = neighbor_locs[i]
            s_next = rng.choice(NUM_STATES, p=T_new[s, :])
            
            # T-free prediction error
            predicted = U_os[s, :] * mu
            err = U_os[s_next, :] - predicted
            
            visit_counts[i] += 1
            H_hat_neighbors[i, :] += (err - H_hat_neighbors[i, :]) / visit_counts[i]
    
    H_hat = np.zeros((num_affected, d))
    H_hat[barrier_idx, :] = psi_R_barrier
    H_hat[neighbor_idx, :] = H_hat_neighbors
    return H_hat


print("\nBarrier-crossing planning (T-free prediction errors):")
print("=" * 65)
for n_visits in [1, 5, 10, 50, 100]:
    H_hat = estimate_H_Tfree(
        T_new, U_os, mu, neighbor_locs,
        psi_R_barrier, barrier_idx, neighbor_idx,
        num_affected, d, n_visits_per_state=n_visits
    )
    M_hat = compute_M_from_H(H_hat, phi_C, d)
    mean_s, med_s, suc_s = evaluate_barrier_crossing(
        U_os, M_hat, T_new, barrier_states, ROWS, COLS)
    h_err = np.linalg.norm(H_hat - psi_R_exact) / np.linalg.norm(psi_R_exact)
    
    print(f"  {n_visits:>3d} visits/state: mean={mean_s:.1f}, median={med_s:.1f}, "
          f"success={suc_s:.2f}, H_err={h_err:.4f}")

Eigenvalue range: [0.343899, 9.594643]
Number of negative eigenvalues: 0
Number of eigenvalues < 1e-6:   0
D_os reconstruction error:       7.70e-02

||H_with_T - H_exact||:    3.77e-16
||H_without_T - H_exact||: 1.22e-02

Barrier-crossing planning (T-free prediction errors):
    1 visits/state: mean=18.5, median=18.0, success=1.00, H_err=1.4903
    5 visits/state: mean=18.4, median=18.0, success=1.00, H_err=0.6455
   10 visits/state: mean=18.3, median=18.0, success=1.00, H_err=0.4137
   50 visits/state: mean=18.3, median=18.0, success=1.00, H_err=0.2166
  100 visits/state: mean=18.3, median=18.0, success=1.00, H_err=0.1323


# Version 5 : Bellman loss approach

In [23]:
valid_states = np.array([s for s in range(NUM_STATES) if s not in barrier_states])

def train_M_from_transitions(U_os, T_new, ec, d, valid_states,
                              n_total_samples=5000, lr=0.001, 
                              log_every=100, seed=42):
    N = U_os.shape[0]
    M = np.eye(d)
    rng = np.random.RandomState(seed)
    
    history = {
        'loss': [],
        'M_error': [],
        'sample': [],
    }
    
    running_loss = 0.0
    
    for t in range(n_total_samples):
        s = rng.choice(valid_states)
        s_prime = rng.choice(N, p=T_new[s, :])
        
        f_s = U_os[s, :]
        f_sp = U_os[s_prime, :]
        
        residual = f_s + f_sp @ M - ec * f_s @ M
        loss = np.sum(residual ** 2)
        running_loss = 0.99 * running_loss + 0.01 * loss
        
        grad_M = 2.0 * np.outer(f_sp - ec * f_s, residual)
        M -= lr * grad_M
        
        if (t + 1) % log_every == 0:
            history['loss'].append(running_loss)
            history['M_error'].append(np.linalg.norm(M - M_true) / np.linalg.norm(M_true))
            history['sample'].append(t + 1)
    
    return M, history


print("Learning M from random transitions:")
print("=" * 65)

M_learned, hist = train_M_from_transitions(
    U_os, T_new, ec, d, valid_states,
    n_total_samples=20000, lr=0.0001, log_every=1000
)

for i in range(len(hist['sample'])):
    print(f"  {hist['sample'][i]:>6d} samples: loss={hist['loss'][i]:.4f}, "
          f"M_err={hist['M_error'][i]:.4f}")

# === Evaluate planning ===
print("\nPlanning evaluation:")
mean_s, med_s, suc_s = evaluate_barrier_crossing(
    U_os, M_learned, T_new, barrier_states, ROWS, COLS)
print(f"  Learned M:     mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")

mean_gt, med_gt, suc_gt = evaluate_barrier_crossing(
    U_os, M_true, T_new, barrier_states, ROWS, COLS)
print(f"  Exact M:       mean={mean_gt:.1f}, median={med_gt:.1f}, success={suc_gt:.2f}")

mean_nr, med_nr, suc_nr = evaluate_barrier_crossing(
    U_os, M_identity, T_new, barrier_states, ROWS, COLS)
print(f"  No repair:     mean={mean_nr:.1f}, median={med_nr:.1f}, success={suc_nr:.2f}")

Learning M from random transitions:
    1000 samples: loss=1.2834, M_err=0.1295
    2000 samples: loss=1.2861, M_err=0.1295
    3000 samples: loss=1.2774, M_err=0.1295
    4000 samples: loss=1.2800, M_err=0.1295
    5000 samples: loss=1.2845, M_err=0.1295
    6000 samples: loss=1.2800, M_err=0.1295
    7000 samples: loss=1.2818, M_err=0.1295
    8000 samples: loss=1.2797, M_err=0.1295
    9000 samples: loss=1.2769, M_err=0.1295
   10000 samples: loss=1.2789, M_err=0.1295
   11000 samples: loss=1.2728, M_err=0.1295
   12000 samples: loss=1.2803, M_err=0.1295
   13000 samples: loss=1.2747, M_err=0.1295
   14000 samples: loss=1.2719, M_err=0.1295
   15000 samples: loss=1.2725, M_err=0.1295
   16000 samples: loss=1.2735, M_err=0.1295
   17000 samples: loss=1.2759, M_err=0.1295
   18000 samples: loss=1.2664, M_err=0.1296
   19000 samples: loss=1.2670, M_err=0.1296
   20000 samples: loss=1.2697, M_err=0.1296

Planning evaluation:
  Learned M:     mean=184.5, median=23.0, success=0.66
  Exact

In [24]:
# === Sweep learning rates ===

print("Learning rate sweep (10000 samples each):")
print("=" * 65)

for lr in [0.001, 0.01, 0.1]:
    M_learned, hist = train_M_from_transitions(
        U_os, T_new, ec, d, valid_states,
        n_total_samples=10000, lr=lr, log_every=2000
    )
    
    mean_s, med_s, suc_s = evaluate_barrier_crossing(
        U_os, M_learned, T_new, barrier_states, ROWS, COLS)
    
    print(f"\n  lr={lr}:")
    for i in range(len(hist['sample'])):
        print(f"    {hist['sample'][i]:>6d} samples: loss={hist['loss'][i]:.4f}, "
              f"M_err={hist['M_error'][i]:.4f}")
    print(f"    Planning: mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")

Learning rate sweep (10000 samples each):

  lr=0.001:
      2000 samples: loss=1.2704, M_err=0.1298
      4000 samples: loss=1.2487, M_err=0.1306
      6000 samples: loss=1.2319, M_err=0.1322
      8000 samples: loss=1.2202, M_err=0.1342
     10000 samples: loss=1.2020, M_err=0.1368
    Planning: mean=182.2, median=23.0, success=0.66

  lr=0.01:
      2000 samples: loss=1.1450, M_err=0.1660
      4000 samples: loss=1.0460, M_err=0.2143
      6000 samples: loss=0.9601, M_err=0.2613
      8000 samples: loss=0.9393, M_err=0.3012
     10000 samples: loss=0.8867, M_err=0.3357
    Planning: mean=134.1, median=21.0, success=0.76

  lr=0.1:
      2000 samples: loss=0.9790, M_err=0.5308
      4000 samples: loss=1.0393, M_err=0.5708
      6000 samples: loss=0.9466, M_err=0.5976
      8000 samples: loss=0.9788, M_err=0.6173
     10000 samples: loss=0.9714, M_err=0.6284
    Planning: mean=37.9, median=19.0, success=0.96


In [25]:
# === Longer run at lr=0.1 + mini-batch variant ===

# First: longer SGD run
print("Longer SGD run (lr=0.1, 100k samples):")
print("=" * 65)
M_learned, hist = train_M_from_transitions(
    U_os, T_new, ec, d, valid_states,
    n_total_samples=100000, lr=0.1, log_every=10000
)
for i in range(len(hist['sample'])):
    print(f"  {hist['sample'][i]:>6d} samples: loss={hist['loss'][i]:.4f}, "
          f"M_err={hist['M_error'][i]:.4f}")

mean_s, med_s, suc_s = evaluate_barrier_crossing(
    U_os, M_learned, T_new, barrier_states, ROWS, COLS)
print(f"  Planning: mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")


# Second: mini-batch version
def train_M_minibatch(U_os, T_new, ec, d, valid_states,
                       n_epochs=100, batch_size=128, lr=0.01, 
                       log_every=10, seed=42):
    N = U_os.shape[0]
    M = np.eye(d)
    rng = np.random.RandomState(seed)
    
    history = {'loss': [], 'M_error': [], 'epoch': []}
    total_samples = 0
    
    for epoch in range(n_epochs):
        # Sample a batch
        states = rng.choice(valid_states, size=batch_size)
        nexts = np.array([rng.choice(N, p=T_new[s, :]) for s in states])
        
        # Batch gradient
        grad_M = np.zeros((d, d))
        batch_loss = 0.0
        
        for s, sp in zip(states, nexts):
            f_s = U_os[s, :]
            f_sp = U_os[sp, :]
            residual = f_s + f_sp @ M - ec * f_s @ M
            grad_M += 2.0 * np.outer(f_sp - ec * f_s, residual)
            batch_loss += np.sum(residual ** 2)
        
        grad_M /= batch_size
        batch_loss /= batch_size
        M -= lr * grad_M
        total_samples += batch_size
        
        if (epoch + 1) % log_every == 0:
            m_err = np.linalg.norm(M - M_true) / np.linalg.norm(M_true)
            history['loss'].append(batch_loss)
            history['M_error'].append(m_err)
            history['epoch'].append(epoch + 1)
    
    return M, history, total_samples

print("\n\nMini-batch training (batch=128):")
print("=" * 65)

for lr in [0.01, 0.1]:
    M_mb, hist_mb, total = train_M_minibatch(
        U_os, T_new, ec, d, valid_states,
        n_epochs=500, batch_size=128, lr=lr, log_every=100
    )
    
    mean_s, med_s, suc_s = evaluate_barrier_crossing(
        U_os, M_mb, T_new, barrier_states, ROWS, COLS)
    
    print(f"\n  lr={lr} ({total} total samples):")
    for i in range(len(hist_mb['epoch'])):
        print(f"    epoch {hist_mb['epoch'][i]:>4d}: loss={hist_mb['loss'][i]:.4f}, "
              f"M_err={hist_mb['M_error'][i]:.4f}")
    print(f"    Planning: mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")

Longer SGD run (lr=0.1, 100k samples):
   10000 samples: loss=0.9714, M_err=0.6284
   20000 samples: loss=1.0142, M_err=0.6824
   30000 samples: loss=0.9911, M_err=0.7231
   40000 samples: loss=0.9819, M_err=0.7432
   50000 samples: loss=1.0535, M_err=0.7523
   60000 samples: loss=0.9719, M_err=0.7653
   70000 samples: loss=1.0317, M_err=0.7655
   80000 samples: loss=0.9733, M_err=0.7684
   90000 samples: loss=1.0262, M_err=0.7727
  100000 samples: loss=0.9921, M_err=0.7758
  Planning: mean=78.6, median=20.0, success=0.88


Mini-batch training (batch=128):

  lr=0.01 (64000 total samples):
    epoch  100: loss=1.2817, M_err=0.1295
    epoch  200: loss=1.2672, M_err=0.1296
    epoch  300: loss=1.2520, M_err=0.1299
    epoch  400: loss=1.2504, M_err=0.1303
    epoch  500: loss=1.2413, M_err=0.1310
    Planning: mean=182.2, median=23.0, success=0.66

  lr=0.1 (64000 total samples):
    epoch  100: loss=1.2075, M_err=0.1367
    epoch  200: loss=1.1297, M_err=0.1547
    epoch  300: loss=1.0

In [26]:
# === Thorough evaluation of Bellman loss approach ===

def train_M_bellman(U_os, T_new, ec, d, valid_states,
                    n_total_samples=500000, lr_init=0.1, lr_decay=0.9999,
                    log_every=50000, seed=42):
    """
    Learn M from sampled transitions with decaying learning rate.
    L = ||f(s) + f(s') @ M - e^c(s) * f(s) @ M||^2
    """
    N = U_os.shape[0]
    M = np.eye(d)
    rng = np.random.RandomState(seed)
    
    history = {'loss': [], 'M_error': [], 'sample': []}
    running_loss = 0.0
    lr = lr_init
    
    for t in range(n_total_samples):
        s = rng.choice(valid_states)
        s_prime = rng.choice(N, p=T_new[s, :])
        
        f_s = U_os[s, :]
        f_sp = U_os[s_prime, :]
        
        residual = f_s + f_sp @ M - ec * f_s @ M
        loss = np.sum(residual ** 2)
        running_loss = 0.99 * running_loss + 0.01 * loss
        
        grad_M = 2.0 * np.outer(f_sp - ec * f_s, residual)
        M -= lr * grad_M
        lr *= lr_decay
        
        if (t + 1) % log_every == 0:
            m_err = np.linalg.norm(M - M_true) / np.linalg.norm(M_true)
            history['loss'].append(running_loss)
            history['M_error'].append(m_err)
            history['sample'].append(t + 1)
    
    return M, history


# === Run at multiple scales ===
print("Advisor's Bellman loss approach")
print("=" * 70)

for n_samples, lr_init, lr_decay in [
    (100000,  0.1,  1.0),       # constant lr
    (500000,  0.1,  1.0),       # more samples, constant lr
    (500000,  0.1,  0.99999),   # decaying lr
    (500000,  0.01, 1.0),       # smaller constant lr, more samples
]:
    M_learned, hist = train_M_bellman(
        U_os, T_new, ec, d, valid_states,
        n_total_samples=n_samples, lr_init=lr_init, lr_decay=lr_decay,
        log_every=n_samples // 5
    )
    
    mean_s, med_s, suc_s = evaluate_barrier_crossing(
        U_os, M_learned, T_new, barrier_states, ROWS, COLS)
    
    print(f"\n  n={n_samples}, lr={lr_init}, decay={lr_decay}:")
    for i in range(len(hist['sample'])):
        print(f"    {hist['sample'][i]:>7d}: loss={hist['loss'][i]:.4f}, M_err={hist['M_error'][i]:.4f}")
    print(f"    Planning: mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")

# === Summary comparison ===
print("\n" + "=" * 70)
print("Summary comparison:")
print(f"  Exact M (ground truth):        mean=18.3, median=18.0, success=1.00")
print(f"  No repair:                     mean=184.5, median=23.0, success=0.66")
print(f"  Prediction error (22 samples): mean=18.5, median=18.0, success=1.00")

Advisor's Bellman loss approach

  n=100000, lr=0.1, decay=1.0:
      20000: loss=1.0142, M_err=0.6824
      40000: loss=0.9819, M_err=0.7432
      60000: loss=0.9719, M_err=0.7653
      80000: loss=0.9733, M_err=0.7684
     100000: loss=0.9921, M_err=0.7758
    Planning: mean=78.6, median=20.0, success=0.88

  n=500000, lr=0.1, decay=1.0:
     100000: loss=0.9921, M_err=0.7758
     200000: loss=0.9362, M_err=0.7737
     300000: loss=0.9671, M_err=0.7791
     400000: loss=0.9781, M_err=0.7781
     500000: loss=0.9562, M_err=0.7736
    Planning: mean=97.4, median=20.0, success=0.83

  n=500000, lr=0.1, decay=0.99999:
     100000: loss=0.8514, M_err=0.7263
     200000: loss=0.7953, M_err=0.7181
     300000: loss=0.7911, M_err=0.7143
     400000: loss=0.7890, M_err=0.7132
     500000: loss=0.7929, M_err=0.7125
    Planning: mean=23.2, median=19.0, success=0.99

  n=500000, lr=0.01, decay=1.0:
     100000: loss=0.8120, M_err=0.5554
     200000: loss=0.7967, M_err=0.6204
     300000: loss=0

# Version 6 : Iterative Bellman loss

In [27]:
# === Combined Bellman + Eq 16 approach ===

def train_combined(U_os, T_new, ec, d, valid_states,
                   phi_C, delta_Phi, psi_R_barrier,
                   barrier_idx, neighbor_idx, num_affected,
                   w_true, M_true,
                   n_outer=10, bellman_samples_per_round=10000,
                   bellman_lr=0.1, eq16_iters=200, eq16_lr=0.01,
                   seed=42):
    """
    Alternating:
    1. Run Bellman SGD on M for a batch of transitions
    2. Fix M, optimize W via Eq 16: ||I + GH(W)M - M||^2
    3. Rebuild M from W via Woodbury: M = I + G(I - HG)^{-1}H
    4. Repeat
    """
    N = U_os.shape[0]
    M = np.eye(d)
    theta = np.zeros(len(w_true))
    rng = np.random.RandomState(seed)
    total_samples = 0
    
    history = {
        'w_error': [], 'M_error': [], 'planning_steps': [],
        'total_samples': [], 'stage': []
    }
    
    for outer in range(n_outer):
        # === Stage 1: Bellman updates on M ===
        for t in range(bellman_samples_per_round):
            s = rng.choice(valid_states)
            s_prime = rng.choice(N, p=T_new[s, :])
            
            f_s = U_os[s, :]
            f_sp = U_os[s_prime, :]
            
            residual = f_s + f_sp @ M - ec * f_s @ M
            grad_M = 2.0 * np.outer(f_sp - ec * f_s, residual)
            M -= bellman_lr * grad_M
        
        total_samples += bellman_samples_per_round
        
        # Log after Bellman
        m_err = np.linalg.norm(M - M_true) / np.linalg.norm(M_true)
        w_err = np.linalg.norm(np.tanh(theta) - w_true)
        history['M_error'].append(m_err)
        history['w_error'].append(w_err)
        history['total_samples'].append(total_samples)
        history['stage'].append('bellman')
        
        print(f"  Round {outer+1} after Bellman ({total_samples} samples): "
              f"M_err={m_err:.4f}, w_err={w_err:.4f}")
        
        # === Stage 2: Fix M, optimize W via Eq 16 ===
        for it in range(eq16_iters):
            loss, grad = compute_loss_and_grad_fixed_M(
                theta, M, phi_C, delta_Phi, psi_R_barrier,
                barrier_idx, neighbor_idx, num_affected, d
            )
            theta -= eq16_lr * grad
        
        w_learned = np.tanh(theta)
        w_err = np.linalg.norm(w_learned - w_true)
        
        print(f"           after Eq 16: w_err={w_err:.4f}, "
              f"w_mean={np.mean(w_learned):.4f}, loss={loss:.2e}")
        
        # === Stage 3: Rebuild M from W via Woodbury ===
        psi_R_learned = build_psi_R(w_learned, delta_Phi, psi_R_barrier,
                                     barrier_idx, neighbor_idx, num_affected, d)
        M = compute_M_from_H(psi_R_learned, phi_C, d)
        
        m_err = np.linalg.norm(M - M_true) / np.linalg.norm(M_true)
        history['M_error'].append(m_err)
        history['w_error'].append(w_err)
        history['total_samples'].append(total_samples)
        history['stage'].append('rebuild')
        
        # Evaluate planning
        mean_s, med_s, suc_s = evaluate_barrier_crossing(
            U_os, M, T_new, barrier_states, ROWS, COLS)
        history['planning_steps'].append(mean_s)
        
        print(f"           after rebuild: M_err={m_err:.4f}, "
              f"planning: mean={mean_s:.1f}, median={med_s:.1f}, success={suc_s:.2f}")
        print()
    
    return M, theta, history


# === Run combined approach ===
print("Combined Bellman + Eq 16 approach")
print("=" * 70)

M_combined, theta_combined, hist_combined = train_combined(
    U_os, T_new, ec, d, valid_states,
    phi_C, delta_Phi, psi_R_barrier,
    barrier_idx, neighbor_idx, num_affected,
    w_true, M_true,
    n_outer=10, bellman_samples_per_round=10000,
    bellman_lr=0.1, eq16_iters=200, eq16_lr=0.01
)

w_combined = np.tanh(theta_combined)
print(f"\nFinal W: {np.round(w_combined[:5], 4)} (true: {w_true[:5]})")
print(f"\nComparison:")
print(f"  Exact M:                       mean=18.3, median=18.0, success=1.00")
print(f"  No repair:                     mean=184.5, median=23.0, success=0.66")
print(f"  Prediction error (22 samples): mean=18.5, median=18.0, success=1.00")

Combined Bellman + Eq 16 approach
  Round 1 after Bellman (10000 samples): M_err=0.6284, w_err=1.1726
           after Eq 16: w_err=0.9032, w_mean=0.0708, loss=1.86e+02
           after rebuild: M_err=0.0826, planning: mean=18.3, median=18.0, success=1.00

  Round 2 after Bellman (20000 samples): M_err=0.6183, w_err=0.9032
           after Eq 16: w_err=0.5355, w_mean=0.1806, loss=1.76e+02
           after rebuild: M_err=0.0481, planning: mean=18.3, median=18.0, success=1.00

  Round 3 after Bellman (30000 samples): M_err=0.6228, w_err=0.5355
           after Eq 16: w_err=0.7179, w_mean=0.1233, loss=1.75e+02
           after rebuild: M_err=0.0670, planning: mean=18.3, median=18.0, success=1.00

  Round 4 after Bellman (40000 samples): M_err=0.6205, w_err=0.7179
           after Eq 16: w_err=0.4399, w_mean=0.1821, loss=1.78e+02
           after rebuild: M_err=0.0426, planning: mean=18.3, median=18.0, success=1.00

  Round 5 after Bellman (50000 samples): M_err=0.6151, w_err=0.4399
      

In [28]:
# === Test: drop barrier states from Woodbury, only use neighbor states ===

# Build C and phi_C using only neighbor states
C_neighbors = np.eye(NUM_STATES)[:, delta_locs[neighbor_idx]]
phi_C_neighbors = U_os.T @ C_neighbors  # (d, n_neighbors)

# H from exact knowledge (for comparison)
psi_R_neighbors = psi_R_exact[neighbor_idx, :]  # (n_neighbors, d)

# M using only neighbor states
M_neighbors_only = compute_M_from_H(psi_R_neighbors, phi_C_neighbors, d)

# M using all states (our previous ground truth)
M_all = compute_M_from_H(psi_R_exact, phi_C, d)

print("Planning comparison: neighbors-only vs all affected states")
print("=" * 65)

mean_all, med_all, suc_all = evaluate_barrier_crossing(
    U_os, M_all, T_new, barrier_states, ROWS, COLS)
print(f"  All affected states (B={num_affected}):  mean={mean_all:.1f}, "
      f"median={med_all:.1f}, success={suc_all:.2f}")

mean_nb, med_nb, suc_nb = evaluate_barrier_crossing(
    U_os, M_neighbors_only, T_new, barrier_states, ROWS, COLS)
print(f"  Neighbors only (B={n_neighbors}):       mean={mean_nb:.1f}, "
      f"median={med_nb:.1f}, success={suc_nb:.2f}")

mean_nr, med_nr, suc_nr = evaluate_barrier_crossing(
    U_os, np.eye(d), T_new, barrier_states, ROWS, COLS)
print(f"  No repair:                       mean={mean_nr:.1f}, "
      f"median={med_nr:.1f}, success={suc_nr:.2f}")

print(f"\n  M error (neighbors-only vs all): "
      f"{np.linalg.norm(M_neighbors_only - M_all) / np.linalg.norm(M_all):.4f}")

Planning comparison: neighbors-only vs all affected states
  All affected states (B=32):  mean=18.3, median=18.0, success=1.00
  Neighbors only (B=22):       mean=18.3, median=18.0, success=1.00
  No repair:                       mean=184.5, median=23.0, success=0.66

  M error (neighbors-only vs all): 0.0884
